# Phase 1 — SERENGETI baselines



## 1. Environment

In [ ]:
%pip install -q datasets accelerate

In [ ]:
import gc
import json
import os
import random
import shutil
import warnings

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from datasets import load_dataset
from google.colab import drive
from sklearn.metrics import f1_score
from torch.cuda.amp import GradScaler, autocast
from torch.optim import AdamW
from torch.utils.data import DataLoader, Dataset
from tqdm.auto import tqdm
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup

warnings.filterwarnings("ignore")
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

drive.mount("/content/drive")

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

SAVE_DIR = "/content/drive/MyDrive/EmotionDetection/CAST_checkpoints"
PHASE1_DIR = f"{SAVE_DIR}/Phase1"
SERENGETI_DIR = f"{PHASE1_DIR}/serengeti"
CKPT_DIR = f"{SERENGETI_DIR}/ckpt"
RESULTS_PATH = f"{SERENGETI_DIR}/phase1_serengeti.json"
PRED_DIR = f"{PHASE1_DIR}/predictions"
SUMMARY_DIR = f"{PHASE1_DIR}/summary"
CACHE_DIR = f"{SAVE_DIR}/data_cache"

for directory in [
    PHASE1_DIR,
    SERENGETI_DIR,
    CKPT_DIR,
    PRED_DIR,
    SUMMARY_DIR,
    CACHE_DIR,
]:
    os.makedirs(directory, exist_ok=True)

RUN_MODEL_TRAINING = False
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

MODEL_CACHE_DIR = f"{SAVE_DIR}/model_cache_serengeti"
LOCAL_MODEL_CACHE = "/content/hf_cache_local_serengeti"
os.makedirs(MODEL_CACHE_DIR, exist_ok=True)
os.makedirs(LOCAL_MODEL_CACHE, exist_ok=True)
os.environ["HF_HOME"] = MODEL_CACHE_DIR
os.environ["TRANSFORMERS_CACHE"] = f"{MODEL_CACHE_DIR}/hub"
os.environ["HF_DATASETS_CACHE"] = f"{MODEL_CACHE_DIR}/datasets"
os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_DISABLE_SYMLINKS"] = "1"

DATASET_HF_NAME = "brighter-dataset/BRIGHTER-emotion-categories"
MODEL_NAME = "UBC-NLP/serengeti-E250"
LR = 2e-5
NUM_EPOCHS = 5
BATCH_SIZE = 8
MAX_LENGTH = 256
PATIENCE = 2

LANGUAGES = {
    "eng": {"name": "English",         "tier": 1, "family": "Indo-European", "subfamily": "Germanic"},
    "hin": {"name": "Hindi",           "tier": 1, "family": "Indo-European", "subfamily": "Indo-Aryan"},
    "rus": {"name": "Russian",         "tier": 1, "family": "Indo-European", "subfamily": "Slavic"},
    "hau": {"name": "Hausa",           "tier": 2, "family": "Afroasiatic",   "subfamily": "Chadic"},
    "kin": {"name": "Kinyarwanda",     "tier": 2, "family": "Niger-Congo",   "subfamily": "Bantu (Great Lakes)"},
    "sun": {"name": "Sundanese",       "tier": 2, "family": "Austronesian",  "subfamily": "Sundic"},
    "yor": {"name": "Yoruba",          "tier": 3, "family": "Niger-Congo",   "subfamily": "Yoruboid"},
    "vmw": {"name": "Emakhuwa",        "tier": 3, "family": "Niger-Congo",   "subfamily": "Bantu (Makua-Lomwe)"},
    "pcm": {"name": "Nigerian Pidgin", "tier": 3, "family": "Creole",        "subfamily": "English-Based"},
}
LANG_ORDER = ["eng", "hin", "rus", "hau", "kin", "sun", "yor", "vmw", "pcm"]
EMOTION_ORDER = ["anger", "disgust", "fear", "joy", "sadness", "surprise"]
ABSENT_EMOTIONS = {"eng": {"disgust"}}


def save_predictions(phase, lang, condition, y_true, y_pred, emotions):
    path = f"{PRED_DIR}/{phase}_{lang}_{condition}.json"
    atomic_json_write(
        path,
        {
            "emotions": list(emotions),
            "y_true": np.asarray(y_true).astype(int).tolist(),
            "y_pred": np.asarray(y_pred).astype(int).tolist(),
        },
    )


print(f"Device            : {DEVICE}")
print(f"Model             : {MODEL_NAME}")
print(f"Training enabled  : {RUN_MODEL_TRAINING}")
print(f"Results           : {RESULTS_PATH}")

# ── Published benchmark reference ──────────────────────────────────────────────
PUBLISHED_BEST = {
    "track_a": {
        "eng": 0.823, "hin": 0.926, "rus": 0.901,
        "hau": 0.751, "kin": 0.657, "sun": 0.550,
        "yor": 0.461, "vmw": 0.325, "pcm": 0.674,
    },
    "track_c": {
        "eng": 0.797, "hin": 0.919, "rus": 0.906,
        "hau": 0.709, "kin": 0.519, "sun": 0.467,
        "yor": 0.359, "vmw": 0.210, "pcm": 0.674,
    },
}
BENCHMARK_SRC_A = "SemEval-2025 Task 11, Table 5 (Track A)"
BENCHMARK_SRC_C = "SemEval-2025 Task 11, Table 7 (Track C)"

Mounted at /content/drive
Device            : cuda
Model             : UBC-NLP/serengeti-E250
Training enabled  : False
Results           : /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase1/serengeti/phase1_serengeti.json


## 2. Data loading

In [ ]:
def get_emotion_cols(df):
    return [
        emotion
        for emotion in EMOTION_ORDER
        if emotion in df.columns
        and df[emotion].fillna(0).sum() > 0
    ]

def load_split(lang_code, split):
    cache_path = f"{CACHE_DIR}/{lang_code}_{split}.parquet"

    if os.path.exists(cache_path):
        return pd.read_parquet(cache_path)

    dataset = load_dataset(DATASET_HF_NAME, lang_code)
    split_key = split

    if split_key not in dataset:
        split_key = {
            "validation": "dev",
            "dev": "validation",
        }.get(split)

    if not split_key or split_key not in dataset:
        return None

    dataframe = dataset[split_key].to_pandas()

    for emotion in EMOTION_ORDER:
        if emotion not in dataframe.columns:
            dataframe[emotion] = 0

    os.makedirs(CACHE_DIR, exist_ok=True)
    dataframe.to_parquet(cache_path)
    return dataframe

DATA = {}
if RUN_MODEL_TRAINING:
    print("Loading BRIGHTER data ...")
    for code in LANG_ORDER:
        DATA[code] = {}
        for split in ["train", "validation", "test"]:
            dataframe = load_split(code, split)
            if dataframe is not None:
                DATA[code][split] = dataframe

        required_splits = {"train", "validation", "test"}
        missing_splits = sorted(required_splits - set(DATA[code]))
        if missing_splits:
            raise RuntimeError(f"{code}: missing dataset splits {missing_splits}")

        print(
            f"  {code.upper()} {LANGUAGES[code]['name']:18s} "
            f"train={len(DATA[code]['train']):5d} "
            f"validation={len(DATA[code]['validation']):5d} "
            f"test={len(DATA[code]['test']):5d}"
        )
else:
    print("Data loading skipped; completed predictions will be validated.")

Data loading skipped; completed predictions will be validated.


## 3. Model, training and evaluation

In [ ]:
class EmotionDataset(Dataset):
    def __init__(self, df, tokenizer, emotion_cols, max_len=MAX_LENGTH):
        self.texts = df["text"].tolist()
        self.labels = df[emotion_cols].fillna(0).astype(float).values
        self.tok = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        enc = self.tok(
            self.texts[idx],
            max_length=self.max_len,
            padding="max_length",
            truncation=True,
            return_tensors="pt",
        )
        return {
            "input_ids": enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels": torch.tensor(self.labels[idx], dtype=torch.float),
        }


def get_eval_emotions(lang_code, emotions):
    absent = ABSENT_EMOTIONS.get(lang_code, set())
    return [emotion for emotion in emotions if emotion not in absent]

def macro_f1(y_true, y_pred, emotions, lang_code):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    emotions = list(emotions)

    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"{lang_code}: y_true shape {y_true.shape} does not match "
            f"y_pred shape {y_pred.shape}"
        )
    if y_true.ndim != 2 or y_true.shape[1] != len(emotions):
        raise ValueError(
            f"{lang_code}: prediction width {y_true.shape} is inconsistent "
            f"with emotions {emotions}"
        )

    unrounded_scores = {
        emotion: float(
            f1_score(
                y_true[:, index],
                y_pred[:, index],
                zero_division=0,
            )
        )
        for index, emotion in enumerate(emotions)
    }

    scores = {
        emotion: round(score, 4)
        for emotion, score in unrounded_scores.items()
    }

    eval_emotions = get_eval_emotions(lang_code, emotions)
    if not eval_emotions:
        raise ValueError(f"{lang_code}: no active evaluation emotions")

    scores["macro_f1"] = round(
        float(np.mean([unrounded_scores[e] for e in eval_emotions])),
        4,
    )
    return scores


def compute_class_weights(train_df, emotion_cols):
    pos_counts = train_df[emotion_cols].fillna(0).sum()
    total = len(train_df)
    weights = total / (2 * pos_counts.clip(lower=1))
    weights = weights.clip(upper=10)
    return torch.tensor(weights.values, dtype=torch.float)


class SerengetiMultiLabel(nn.Module):
    def __init__(self, model_name, n_labels, dropout=0.1):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(
            model_name,
            cache_dir=LOCAL_MODEL_CACHE,
        )
        hidden = self.encoder.config.hidden_size
        self.drop = nn.Dropout(dropout)
        self.classifier = nn.Linear(hidden, n_labels)

    def forward(self, input_ids, attention_mask):
        out = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
        )
        cls = self.drop(out.last_hidden_state[:, 0, :])
        return self.classifier(cls)


def train_serengeti(
    train_df,
    dev_df,
    tokenizer,
    emotion_cols,
    lang_code,
    ckpt_dir,
    epochs=NUM_EPOCHS,
    batch_size=BATCH_SIZE,
    lr=LR,
    max_len=MAX_LENGTH,
    patience=PATIENCE,
):
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    os.makedirs(ckpt_dir, exist_ok=True)

    train_dl = DataLoader(
        EmotionDataset(train_df, tokenizer, emotion_cols, max_len),
        batch_size=batch_size,
        shuffle=True,
    )
    dev_dl = DataLoader(
        EmotionDataset(dev_df, tokenizer, emotion_cols, max_len),
        batch_size=batch_size * 2,
    )

    print(
        f"      train: {len(train_df)} examples "
        f"({len(train_dl)} batches/epoch)"
    )

    model = SerengetiMultiLabel(
        MODEL_NAME,
        n_labels=len(emotion_cols),
    ).to(DEVICE)
    class_weights = compute_class_weights(
        train_df,
        emotion_cols,
    ).to(DEVICE)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=class_weights)
    optimiser = AdamW(model.parameters(), lr=lr, weight_decay=0.01)
    scheduler = get_linear_schedule_with_warmup(
        optimiser,
        num_warmup_steps=int(0.1 * len(train_dl) * epochs),
        num_training_steps=len(train_dl) * epochs,
    )
    scaler = GradScaler()

    best_path = f"{ckpt_dir}/best.pt"
    state_path = f"{ckpt_dir}/state.json"

    best_f1 = -1.0
    no_improve = 0
    start_epoch = 1

    if os.path.exists(state_path) and os.path.exists(best_path):
        try:
            with open(state_path, encoding="utf-8") as f:
                state = json.load(f)
            start_epoch = state["epoch"] + 1
            best_f1 = state["best_f1"]
            no_improve = state["no_improve"]
            model.load_state_dict(
                torch.load(best_path, map_location=DEVICE)
            )
            print(
                f"      resuming from epoch {start_epoch} "
                f"(best dev F1: {best_f1:.4f})"
            )
        except (json.JSONDecodeError, KeyError, RuntimeError, OSError):
            print("      corrupt checkpoint -- starting fresh")
            start_epoch = 1
            best_f1 = -1.0
            no_improve = 0

    for epoch in range(start_epoch, epochs + 1):
        model.train()
        running_loss = 0.0
        progress = tqdm(
            train_dl,
            desc=f"      epoch {epoch}/{epochs}",
            leave=False,
        )

        for step, batch in enumerate(progress, 1):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)

            optimiser.zero_grad()
            with autocast():
                loss = loss_fn(
                    model(input_ids, attention_mask),
                    labels,
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimiser)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimiser)
            scaler.update()
            scheduler.step()

            running_loss += loss.item()
            progress.set_postfix(
                loss=f"{running_loss / step:.4f}"
            )

        model.eval()
        dev_predictions = []
        dev_labels = []

        with torch.no_grad():
            for batch in dev_dl:
                input_ids = batch["input_ids"].to(DEVICE)
                attention_mask = batch["attention_mask"].to(DEVICE)
                predictions = (
                    torch.sigmoid(model(input_ids, attention_mask)) > 0.5
                ).int().cpu().numpy()

                dev_predictions.append(predictions)
                dev_labels.append(batch["labels"].int().numpy())

        dev_f1 = macro_f1(
            np.vstack(dev_labels),
            np.vstack(dev_predictions),
            emotion_cols,
            lang_code=lang_code,
        )["macro_f1"]

        print(
            f"      epoch {epoch}/{epochs} -- "
            f"loss: {running_loss / len(train_dl):.4f} -- "
            f"dev F1: {dev_f1:.4f}"
        )

        if dev_f1 > best_f1:
            best_f1 = dev_f1
            no_improve = 0
            torch.save(model.state_dict(), best_path)
        else:
            no_improve += 1

        _atomic_json_write(
            state_path,
            {
                "epoch": epoch,
                "best_f1": best_f1,
                "no_improve": no_improve,
            },
        )

        if no_improve >= patience:
            print(f"      early stop (patience={patience})")
            break

    if not os.path.exists(best_path):
        raise RuntimeError(
            f"{lang_code}: training finished without a best checkpoint"
        )

    model.load_state_dict(
        torch.load(best_path, map_location=DEVICE)
    )
    shutil.rmtree(ckpt_dir, ignore_errors=True)
    return model


@torch.no_grad()
def evaluate_serengeti(
    model,
    test_df,
    tokenizer,
    emotion_cols,
    lang_code,
    batch_size=BATCH_SIZE * 2,
    run_tag=None,
):
    test_dl = DataLoader(
        EmotionDataset(test_df, tokenizer, emotion_cols),
        batch_size=batch_size,
    )

    model.eval()
    predictions = []
    labels = []

    for batch in test_dl:
        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        batch_predictions = (
            torch.sigmoid(model(input_ids, attention_mask)) > 0.5
        ).int().cpu().numpy()

        predictions.append(batch_predictions)
        labels.append(batch["labels"].int().numpy())

    y_true = np.vstack(labels)
    y_pred = np.vstack(predictions)

    if run_tag is not None:
        save_predictions(
            run_tag[0],
            run_tag[1],
            run_tag[2],
            y_true,
            y_pred,
            emotion_cols,
        )

    return macro_f1(
        y_true,
        y_pred,
        emotion_cols,
        lang_code=lang_code,
    )

In [ ]:
if RUN_MODEL_TRAINING:
    tokenizer = AutoTokenizer.from_pretrained(
        MODEL_NAME,
        cache_dir=LOCAL_MODEL_CACHE,
    )
    print("SERENGETI tokenizer loaded.")
else:
    tokenizer = None
    print("Model loading skipped.")

Model loading skipped.


## 4. Load and validate saved prediction state

In [ ]:
def prediction_path(track, lang_code):
    if track == "track_a":
        return f"{PRED_DIR}/phase1_serengeti_{lang_code}_mono.json"
    if track == "track_c":
        return f"{PRED_DIR}/phase1_serengeti_c_{lang_code}_loo.json"
    raise ValueError(f"Unknown track: {track}")


def load_prediction_payload(path):
    if not os.path.exists(path):
        raise FileNotFoundError(path)

    with open(path, encoding="utf-8") as handle:
        payload = json.load(handle)

    missing = sorted({"emotions", "y_true", "y_pred"} - set(payload))
    if missing:
        raise ValueError(f"{path}: missing keys {missing}")

    emotions = list(payload["emotions"])
    y_true = np.asarray(payload["y_true"], dtype=int)
    y_pred = np.asarray(payload["y_pred"], dtype=int)

    if len(emotions) != len(set(emotions)):
        raise ValueError(f"{path}: duplicated emotion names")
    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"{path}: y_true shape {y_true.shape} != y_pred shape {y_pred.shape}"
        )
    if y_true.ndim != 2 or y_true.shape[1] != len(emotions):
        raise ValueError(
            f"{path}: shape {y_true.shape} is inconsistent with {emotions}"
        )
    if len(y_true) == 0:
        raise ValueError(f"{path}: no prediction rows")

    return emotions, y_true, y_pred


def valid_saved_prediction(track, lang_code):
    try:
        load_prediction_payload(prediction_path(track, lang_code))
        return True
    except (FileNotFoundError, json.JSONDecodeError, OSError, TypeError, ValueError):
        return False


if os.path.exists(RESULTS_PATH):
    with open(RESULTS_PATH, encoding="utf-8") as handle:
        serengeti_results = json.load(handle)
else:
    serengeti_results = {}

serengeti_results.setdefault("track_a", {})
serengeti_results.setdefault("track_c", {})

missing_conditions = [
    f"{track}/{code}"
    for track in ["track_a", "track_c"]
    for code in LANG_ORDER
    if not valid_saved_prediction(track, code)
]

if missing_conditions and not RUN_MODEL_TRAINING:
    raise RuntimeError(
        "Training is disabled and these prediction conditions are missing or "
        f"invalid: {missing_conditions}"
    )

print(f"Prediction conditions requiring generation: {missing_conditions or 'none'}")

Prediction conditions requiring generation: none


## 5. Track A - monolingual fine-tuning

In [ ]:
if RUN_MODEL_TRAINING:
    for code in LANG_ORDER:
        if valid_saved_prediction("track_a", code):
            print(f"  Skipping {LANGUAGES[code]['name']} Track A")
            continue

        lang_seed = RANDOM_SEED + LANG_ORDER.index(code)
        random.seed(lang_seed)
        np.random.seed(lang_seed)
        torch.manual_seed(lang_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(lang_seed)

        emotion_cols = get_emotion_cols(DATA[code]["train"])
        print(
            f"  {code.upper()} {LANGUAGES[code]['name']} "
            f"emotions={emotion_cols} seed={lang_seed}"
        )

        model = train_serengeti(
            DATA[code]["train"],
            DATA[code]["validation"],
            tokenizer,
            emotion_cols,
            lang_code=code,
            ckpt_dir=f"{CKPT_DIR}/{code}",
        )
        result = evaluate_serengeti(
            model,
            DATA[code]["test"],
            tokenizer,
            emotion_cols,
            lang_code=code,
            run_tag=("phase1_serengeti", code, "mono"),
        )
        serengeti_results["track_a"][code] = result
        atomic_json_write(RESULTS_PATH, serengeti_results)

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(f"    macro-F1={result['macro_f1']:.4f}")
else:
    print("Track A training skipped.")

Track A training skipped.


## 6. Track C - leave-one-out cross-lingual fine-tuning

In [ ]:
if RUN_MODEL_TRAINING:
    for code in LANG_ORDER:
        if valid_saved_prediction("track_c", code):
            print(f"  Skipping {LANGUAGES[code]['name']} Track C")
            continue

        source_codes = [source for source in LANG_ORDER if source != code]
        train_df = pd.concat(
            [DATA[source]["train"] for source in source_codes],
            ignore_index=True,
        )
        validation_df = pd.concat(
            [DATA[source]["validation"] for source in source_codes],
            ignore_index=True,
        )
        emotion_cols = get_emotion_cols(train_df)

        lang_seed = RANDOM_SEED + 100 + LANG_ORDER.index(code)
        random.seed(lang_seed)
        np.random.seed(lang_seed)
        torch.manual_seed(lang_seed)
        if torch.cuda.is_available():
            torch.cuda.manual_seed_all(lang_seed)

        print(
            f"  {code.upper()} {LANGUAGES[code]['name']} "
            f"sources={source_codes} seed={lang_seed}"
        )

        model = train_serengeti(
            train_df,
            validation_df,
            tokenizer,
            emotion_cols,
            lang_code=code,
            ckpt_dir=f"{CKPT_DIR}/{code}_track_c",
        )
        result = evaluate_serengeti(
            model,
            DATA[code]["test"],
            tokenizer,
            emotion_cols,
            lang_code=code,
            run_tag=("phase1_serengeti_c", code, "loo"),
        )
        serengeti_results["track_c"][code] = result
        atomic_json_write(RESULTS_PATH, serengeti_results)

        del model
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

        print(f"    macro-F1={result['macro_f1']:.4f}")
else:
    print("Track C training skipped.")

Track C training skipped.


## 7. Recalculate, validate and integrate final results

In [ ]:
verified_results = {"track_a": {}, "track_c": {}}

for track in ["track_a", "track_c"]:
    for code in LANG_ORDER:
        path = prediction_path(track, code)
        emotions, y_true, y_pred = load_prediction_payload(path)
        verified_results[track][code] = macro_f1(
            y_true,
            y_pred,
            emotions,
            lang_code=code,
        )

serengeti_results = verified_results
atomic_json_write(RESULTS_PATH, serengeti_results)

for track in ["track_a", "track_c"]:
    missing = [code for code in LANG_ORDER if code not in serengeti_results[track]]
    if missing:
        raise ValueError(f"{track}: missing SERENGETI results for {missing}")

assert serengeti_results["track_a"]["eng"]["macro_f1"] == 0.6460
assert serengeti_results["track_c"]["eng"]["macro_f1"] == 0.3856

phase1_summary_path = f"{SUMMARY_DIR}/phase1_raw_results_gemma4.json"
if not os.path.exists(phase1_summary_path):
    raise FileNotFoundError(
        "Run the cleaned main Phase 1 notebook before integrating SERENGETI: "
        f"{phase1_summary_path}"
    )

with open(phase1_summary_path, encoding="utf-8") as handle:
    phase1_summary = json.load(handle)

phase1_summary["serengeti"] = serengeti_results
atomic_json_write(phase1_summary_path, phase1_summary)

comparison_rows = []
for track in ["track_a", "track_c"]:
    for code in LANG_ORDER:
        xlmr_score = (
            phase1_summary
            .get("xlmr", {})
            .get(track, {})
            .get(code, {})
            .get("macro_f1")
        )
        serengeti_score = serengeti_results[track][code]["macro_f1"]
        _is_a     = (track == "track_a")
        _tl       = "Track A" if _is_a else "Track C"
        _pub_a    = PUBLISHED_BEST["track_a"][code]
        _pub_c    = PUBLISHED_BEST["track_c"][code]
        _pub_this = _pub_a if _is_a else _pub_c
        _src      = BENCHMARK_SRC_A if _is_a else BENCHMARK_SRC_C
        _delta    = round(serengeti_score - xlmr_score, 4) if xlmr_score is not None else None
        comparison_rows.append({
            "Evaluation track":                            _tl,
            "Language code":                               code.upper(),
            "Language":                                    LANGUAGES[code]["name"],
            "XLM-R macro-F1":                              xlmr_score,
            "SERENGETI macro-F1":                          serengeti_score,
            "Backbone effect (SERENGETI minus XLM-R)":    _delta,
            "Published Track A best macro-F1":             _pub_a,
            "Published Track C best macro-F1":             _pub_c,
            "Published best macro-F1 for this row's track": _pub_this,
            "XLM-R gap to same-track published best":      round(xlmr_score - _pub_this, 4) if xlmr_score is not None else None,
            "SERENGETI gap to same-track published best":  round(serengeti_score - _pub_this, 4),
            "Published benchmark source":                  _src,
        })

comparison = pd.DataFrame(comparison_rows)
display(comparison)
comparison.to_csv(f"{SUMMARY_DIR}/phase1_xlmr_vs_serengeti_comparison.csv", index=False)

print(f"Standalone SERENGETI results: {RESULTS_PATH}")
print(f"Integrated Phase 1 summary : {phase1_summary_path}")
print(
    "English verified: "
    f"Track A={serengeti_results['track_a']['eng']['macro_f1']:.4f}, "
    f"Track C={serengeti_results['track_c']['eng']['macro_f1']:.4f}"
)

comparison.to_csv(f"{SUMMARY_DIR}/phase1_xlmr_vs_serengeti_comparison.csv", index=False)

,Evaluation track,Language code,Language,XLM-R macro-F1,SERENGETI macro-F1,Backbone effect (SERENGETI minus XLM-R),Published Track A best macro-F1,Published Track C best macro-F1,Published best macro-F1 for this row's track,XLM-R gap to same-track published best,SERENGETI gap to same-track published best,Published benchmark source
0,Track A,ENG,English,0.5523,0.6460,0.0937,0.823,0.797,0.823,-0.2707,-0.1770,"SemEval-2025 Task 11, Table 5 (Track A)"
1,Track A,HIN,Hindi,0.8550,0.5235,-0.3315,0.926,0.919,0.926,-0.0710,-0.4025,"SemEval-2025 Task 11, Table 5 (Track A)"
2,Track A,RUS,Russian,0.8791,0.8320,-0.0471,0.901,0.906,0.901,-0.0219,-0.0690,"SemEval-2025 Task 11, Table 5 (Track A)"
3,Track A,HAU,Hausa,0.6326,0.6966,0.0640,0.751,0.709,0.751,-0.1184,-0.0544,"SemEval-2025 Task 11, Table 5 (Track A)"
4,Track A,KIN,Kinyarwanda,0.3928,0.5427,0.1499,0.657,0.519,0.657,-0.2642,-0.1143,"SemEval-2025 Task 11, Table 5 (Track A)"
5,Track A,SUN,Sundanese,0.2026,0.3360,0.1334,0.550,0.467,0.550,-0.3474,-0.2140,"SemEval-2025 Task 11, Table 5 (Track A)"
6,Track A,YOR,Yoruba,0.1267,0.4061,0.2794,0.461,0.359,0.461,-0.3343,-0.0549,"SemEval-2025 Task 11, Table 5 (Track A)"
7,Track A,VMW,Emakhuwa,0.0380,0.1658,0.1278,0.325,0.210,0.325,-0.2870,-0.1592,"SemEval-2025 Task 11, Table 5 (Track A)"
8,Track A,PCM,Nigerian Pidgin,0.5692,0.5593,-0.0099,0.674,0.674,0.674,-0.1048,-0.1147,"SemEval-2025 Task 11, Table 5 (Track A)"
9,Track C,ENG,English,0.4217,0.3856,-0.0361,0.823,0.797,0.797,-0.3753,-0.4114,"SemEval-2025 Task 11, Table 7 (Track C)"


Standalone SERENGETI results: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase1/serengeti/phase1_serengeti.json
Integrated Phase 1 summary : /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase1/summary/phase1_raw_results_gemma4.json
English verified: Track A=0.6460, Track C=0.3856


In [ ]:
# ── 8. Export SERENGETI Phase 1 summary tables ───────────────────────────────

XLM_SUMMARY_A = f"{SUMMARY_DIR}/phase1_track_a_summary_gemma4.csv"
XLM_SUMMARY_C = f"{SUMMARY_DIR}/phase1_track_c_summary_gemma4.csv"

if not os.path.exists(XLM_SUMMARY_A) or not os.path.exists(XLM_SUMMARY_C):
    raise FileNotFoundError(
        "XLM-R Phase 1 summary CSVs not found. "
        "Run the main Phase 1 notebook first."
    )

xlmr_a = pd.read_csv(XLM_SUMMARY_A)
xlmr_c = pd.read_csv(XLM_SUMMARY_C)

for track_label, xlmr_df, track_key in [
    ("track_a", xlmr_a, "track_a"),
    ("track_c", xlmr_c, "track_c"),
]:
    rows = []
    _is_a17    = (track_label == "track_a")
    _tl17      = "Track A" if _is_a17 else "Track C"
    _pub_col17 = "Published Track A best macro-F1" if _is_a17 else "Published Track C best macro-F1"
    _gap_srng  = f"Gap: SERENGETI minus published {_tl17} best"
    _gap_g4_17 = f"Gap: Gemma-4-31B minus published {_tl17} best"
    _src17     = BENCHMARK_SRC_A if _is_a17 else BENCHMARK_SRC_C
    for code in LANG_ORDER:
        xlmr_row   = xlmr_df[xlmr_df["Language code"] == code.upper()].iloc[0]
        srng_macro = serengeti_results[track_key][code]["macro_f1"]
        pub_best17 = PUBLISHED_BEST[track_key][code]
        g4_f1_17   = xlmr_row["Gemma-4-31B zero-shot macro-F1"]
        rows.append({
            "Language":                       xlmr_row["Language"],
            "Language code":                  xlmr_row["Language code"],
            "Resource tier":                  xlmr_row["Resource tier"],
            "Linguistic subgroup":            xlmr_row["Linguistic subgroup"],
            "Number of evaluated emotions":   xlmr_row["Number of evaluated emotions"],
            "Majority baseline macro-F1":     xlmr_row["Majority baseline macro-F1"],
            "SERENGETI macro-F1":             srng_macro,
            "Gemma-4-31B zero-shot macro-F1": g4_f1_17,
            _pub_col17:                       pub_best17,
            "Evaluation track":               _tl17,
            _gap_srng:                        round(srng_macro - pub_best17, 4),
            _gap_g4_17:                       round(g4_f1_17 - pub_best17, 4),
            "Published benchmark source":     _src17,
        })
    out_df   = pd.DataFrame(rows)
    out_path = f"{SUMMARY_DIR}/phase1_{track_label}_summary_serengeti.csv"
    out_df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}")
    display(out_df)

print("SERENGETI Phase 1 summary export complete.")

Saved: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase1/summary/phase1_track_a_summary_serengeti.csv


,Language,Language code,Resource tier,Linguistic subgroup,Number of evaluated emotions,Majority baseline macro-F1,SERENGETI macro-F1,Gemma-4-31B zero-shot macro-F1,Published Track A best macro-F1,Evaluation track,Gap: SERENGETI minus published Track A best,Gap: Gemma-4-31B minus published Track A best,Published benchmark source
0,English,ENG,1,Germanic,5,0.4491,0.6460,0.6274,0.823,Track A,-0.1770,-0.1956,"SemEval-2025 Task 11, Table 5 (Track A)"
1,Hindi,HIN,1,Indo-Aryan,6,0.2641,0.5235,0.7421,0.926,Track A,-0.4025,-0.1839,"SemEval-2025 Task 11, Table 5 (Track A)"
2,Russian,RUS,1,Slavic,6,0.2618,0.8320,0.8180,0.901,Track A,-0.0690,-0.0830,"SemEval-2025 Task 11, Table 5 (Track A)"
3,Hausa,HAU,2,Chadic,6,0.3121,0.6966,0.5606,0.751,Track A,-0.0544,-0.1904,"SemEval-2025 Task 11, Table 5 (Track A)"
4,Kinyarwanda,KIN,2,Bantu (Great Lakes),6,0.2176,0.5427,0.4213,0.657,Track A,-0.1143,-0.2357,"SemEval-2025 Task 11, Table 5 (Track A)"
5,Sundanese,SUN,2,Sundic,6,0.3340,0.3360,0.6516,0.550,Track A,-0.2140,0.1016,"SemEval-2025 Task 11, Table 5 (Track A)"
6,Yoruba,YOR,3,Yoruboid,6,0.1647,0.4061,0.3479,0.461,Track A,-0.0549,-0.1131,"SemEval-2025 Task 11, Table 5 (Track A)"
7,Emakhuwa,VMW,3,Bantu (Makua-Lomwe),6,0.1626,0.1658,0.0209,0.325,Track A,-0.1592,-0.3041,"SemEval-2025 Task 11, Table 5 (Track A)"
8,Nigerian Pidgin,PCM,3,English-Based,6,0.3566,0.5593,0.5199,0.674,Track A,-0.1147,-0.1541,"SemEval-2025 Task 11, Table 5 (Track A)"


Saved: /content/drive/MyDrive/EmotionDetection/CAST_checkpoints/Phase1/summary/phase1_track_c_summary_serengeti.csv


,Language,Language code,Resource tier,Linguistic subgroup,Number of evaluated emotions,Majority baseline macro-F1,SERENGETI macro-F1,Gemma-4-31B zero-shot macro-F1,Published Track C best macro-F1,Evaluation track,Gap: SERENGETI minus published Track C best,Gap: Gemma-4-31B minus published Track C best,Published benchmark source
0,English,ENG,1,Germanic,5,0.4491,0.3856,0.6274,0.797,Track C,-0.4114,-0.1696,"SemEval-2025 Task 11, Table 7 (Track C)"
1,Hindi,HIN,1,Indo-Aryan,6,0.2641,0.0776,0.7421,0.919,Track C,-0.8414,-0.1769,"SemEval-2025 Task 11, Table 7 (Track C)"
2,Russian,RUS,1,Slavic,6,0.2618,0.4665,0.8180,0.906,Track C,-0.4395,-0.0880,"SemEval-2025 Task 11, Table 7 (Track C)"
3,Hausa,HAU,2,Chadic,6,0.3121,0.4406,0.5606,0.709,Track C,-0.2684,-0.1484,"SemEval-2025 Task 11, Table 7 (Track C)"
4,Kinyarwanda,KIN,2,Bantu (Great Lakes),6,0.2176,0.3965,0.4213,0.519,Track C,-0.1225,-0.0977,"SemEval-2025 Task 11, Table 7 (Track C)"
5,Sundanese,SUN,2,Sundic,6,0.3340,0.2112,0.6516,0.467,Track C,-0.2558,0.1846,"SemEval-2025 Task 11, Table 7 (Track C)"
6,Yoruba,YOR,3,Yoruboid,6,0.1647,0.2447,0.3479,0.359,Track C,-0.1143,-0.0111,"SemEval-2025 Task 11, Table 7 (Track C)"
7,Emakhuwa,VMW,3,Bantu (Makua-Lomwe),6,0.1626,0.1228,0.0209,0.210,Track C,-0.0872,-0.1891,"SemEval-2025 Task 11, Table 7 (Track C)"
8,Nigerian Pidgin,PCM,3,English-Based,6,0.3566,0.3353,0.5199,0.674,Track C,-0.3387,-0.1541,"SemEval-2025 Task 11, Table 7 (Track C)"


SERENGETI Phase 1 summary export complete.
